# Encode embedding node KG trên GPU (Google Colab)

Chạy 4 model ứng viên (`e5-small`, `e5-base`, `vn-embedding`, `bge-m3`) cho text node v1 và v2,
cộng pool mù dev/test, trong **cùng một môi trường** để so sánh công bằng. Vector giữ float32 (tắt TF32).

1. Runtime → Change runtime type → **T4 GPU**.
2. Chạy lần lượt các ô. Ô upload: chọn file `colab_bundle.zip` trong thư mục dự án.
3. Ô cuối tải về `colab_results.zip`; giải nén vào thư mục gốc dự án trên máy
   (ghi đè `output/kg/embedding_benchmark/`), rồi báo lại để chạy các bước so sánh.

Nếu Colab ngắt kết nối giữa chừng: chạy lại từ ô upload; model nào đã xong sẽ phải encode lại
(thư mục `/content` không giữ lại khi runtime bị xóa).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Chưa bật GPU: Runtime → Change runtime type → T4 GPU'

In [ ]:
import os, zipfile
from google.colab import files
os.chdir('/content')
WORK = '/content/work'
os.makedirs(WORK, exist_ok=True)
bundle = '/content/colab_bundle.zip'
if not os.path.exists(bundle):  # chưa upload thì mở hộp chọn file
    files.upload(target_dir='/content')  # chọn colab_bundle.zip
zipfile.ZipFile(bundle).extractall(WORK)
os.chdir(WORK)
!ls legal_kg evaluation output/kg/embedding_text output/kg/embedding_text_v2

In [ ]:
# torch/numpy có sẵn trên Colab; chỉ cài các gói còn thiếu theo requirements-embedding.txt.
!pip install -q "sentence-transformers>=5.3,<6" "transformers>=5.3,<6" "huggingface-hub>=0.34,<2"
!pip freeze > colab_env.txt
import importlib.metadata as m
print({p: m.version(p) for p in ['torch', 'transformers', 'sentence-transformers', 'numpy']})

In [ ]:
import os, subprocess, sys
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
# (model, batch size). max-batch-tokens giới hạn số token đã pad mỗi batch để vừa 15 GB của T4.
MODELS = [('e5-small', 64), ('e5-base', 32), ('vn-embedding', 16), ('bge-m3', 16)]
CORPORA = [('output/kg/embedding_text/node_texts.jsonl', 'output/kg/embedding_benchmark'),
           ('output/kg/embedding_text_v2/node_texts.jsonl', 'output/kg/embedding_benchmark/text-v2')]

def run(*args):
    command = [sys.executable, '-m', 'legal_kg.embedding_benchmark', '--device', 'cuda', '--threads', '2', *args]
    print('$', ' '.join(command[2:]), flush=True)
    result = subprocess.run(command, capture_output=True, text=True)
    print(result.stdout[-1500:])
    if result.returncode != 0:
        print(result.stderr[-4000:])
        raise RuntimeError(f'Failed: {args}')

for model, batch in MODELS:
    for corpus, output in CORPORA:
        run('--model', model, '--batch-size', str(batch), '--max-batch-tokens', '16384',
            '--corpus', corpus, '--output', output)
    for split in ('dev', 'test'):
        # Pool mù: chỉ node ID theo thứ tự chữ cái, không điểm, không metric.
        run('--model', model, '--pool-only', '--questions', f'evaluation/{split}.jsonl')
print('ALL DONE')

In [ ]:
import os
os.chdir(WORK)
!cp colab_env.txt output/kg/embedding_benchmark/colab_env.txt
!rm -f colab_results.zip && zip -qr colab_results.zip output/kg/embedding_benchmark -x '*checkpoint.npz'
!ls -lh colab_results.zip
from google.colab import files
files.download(os.path.join(WORK, 'colab_results.zip'))